In [1]:
ENV["PYTHONPATH"] = "/home/gridsan/aligho/.local/lib/python3.8/site-packages/PyNormaliz-2.15-py3.8-linux-x86_64.egg";

In [2]:
using DelimitedFiles, PyPlot, LinearAlgebra
using Crystalline, Brillouin, MPBUtils, SymmetryBases
using Crystalline: TEST_αβγs, TEST_αβγ, dot, norm
topology_paper_dir = "../TopologyPaper/"
include(topology_paper_dir * "get-freqs-symeigs.jl")
include(topology_paper_dir * "symeigs-from-io.jl");

In [46]:
filtered_sgnums = Int64[] # Space groups with multidimensional irreps at one or more k-points
filtered_sgnums_dict = Dict{Int64, Vector{String}}() # A dictionary storing a space group number as a key and the k-labels corresponding to multidimensional irreps as the value

filtered_sgnums_tr = Int64[] 
filtered_sgnums_dict_tr = Dict{Int64, Vector{String}}() # Same as the above but with time-reversal symmetry 

for sgnum in 1:230 
    sgnum_lgirreps = lgirreps(sgnum)
    good_klabs = String[] 
    for (k_lab, irreps) in sgnum_lgirreps
        good_irrep = false
        for irrep in irreps
            irrep_dim = first(size(first(irrep.matrices)))
            irrep_dim > 1 || continue # Check that irrep dimension is larger than one 
            push!(filtered_sgnums, sgnum)
            good_irrep = true
        end
        good_irrep && push!(good_klabs, k_lab)
    end
    !isempty(good_klabs) && push!(filtered_sgnums_dict, sgnum => good_klabs)
end
unique!(filtered_sgnums);

# Run the same loop as above but with time reversal symmetry 
for sgnum in 1:230 
    sgnum_lgirreps = lgirreps(sgnum)
    good_klabs = String[] 
    for (k_lab, irreps) in sgnum_lgirreps
        good_irrep = false
        for irrep in realify(irreps)
            irrep_dim = first(size(first(irrep.matrices)))
            irrep_dim > 1 || continue # Check that irrep dimension is larger than one 
            push!(filtered_sgnums_tr, sgnum)
            good_irrep = true
        end
        good_irrep && push!(good_klabs, k_lab)
    end
    !isempty(good_klabs) && push!(filtered_sgnums_dict_tr, sgnum => good_klabs)
end
unique!(filtered_sgnums);
unique!(filtered_sgnums_tr);

print("Number of spacegroups with multidimensional irreps (even without time reversal symmetry): ", length(filtered_sgnums), "\n")
print("Number of spacegroups with multidimensional irreps (with time reversal symmetry): ", length(filtered_sgnums_tr), "\n")

Number of spacegroups with multidimensional irreps (even without time reversal symmetry): 175
Number of spacegroups with multidimensional irreps (with time reversal symmetry): 209


In [66]:
sgnum_pairs_primitive = Dict{Int, Vector{Int}}()
sgnum_pairs_primitive_tr = Dict{Int, Vector{Int}}()

for sgnum_1 in filtered_sgnums 
    sgnum_1_group = primitivize(spacegroup(sgnum_1)) # put in primitive setting
    filtered_group_operations = filter(x->iszero(x.translation), sgnum_1_group) # All group operations which are fully symmorphic 
    pg_1 = find_isomorphic_parent_pointgroup(filtered_group_operations)[1] # Find point group isomorphic to the group corresponding to all symmorphic operations 
    sgnum1_vec = Int[] # Vector of candidate subgroups of sgnum_1
    sgnum_1_c_system = crystalsystem(primitivize(directbasis(sgnum_1), centering(sgnum_1))) # Whether the system is cubic, tetragonal, hexagonal, etc 
    for sgnum_2 in 1:230
        sgnum_2_c_system = crystalsystem(primitivize(directbasis(sgnum_2), centering(sgnum_2)))
        (sgnum_2_c_system == sgnum_1_c_system) || continue # Require that the crystal system remain the same
        sgnum_2_group = primitivize(spacegroup(sgnum_2))
        (filter(x->iszero(x.translation), sgnum_2_group) == sgnum_2_group) || continue # Only look at symmorphic space groups
        pg_2 = find_isomorphic_parent_pointgroup(sgnum_2_group)[1]
        (pg_1 == pg_2) || continue 
        push!(sgnum1_vec, sgnum_2)
    end
    !(isempty(sgnum1_vec)) || continue
    any(x -> x in filtered_sgnums, sgnum1_vec) || continue
    push!(sgnum_pairs_primitive, sgnum_1 => sgnum1_vec)
end

# Same loop as above but with time reversal symmetry

for sgnum_1 in filtered_sgnums_tr 
    sgnum_1_group = primitivize(spacegroup(sgnum_1)) # put in primitive setting
    filtered_group_operations = filter(x->iszero(x.translation), sgnum_1_group) # All group operations which are fully symmorphic 
    pg_1 = find_isomorphic_parent_pointgroup(filtered_group_operations)[1] # Find point group isomorphic to the group corresponding to all symmorphic operations 
    sgnum1_vec = Int[] # Vector of candidate subgroups of sgnum_1
    sgnum_1_c_system = crystalsystem(primitivize(directbasis(sgnum_1), centering(sgnum_1))) # Whether the system is cubic, tetragonal, hexagonal, etc 
    for sgnum_2 in 1:230
        sgnum_2_c_system = crystalsystem(primitivize(directbasis(sgnum_2), centering(sgnum_2)))
        (sgnum_2_c_system == sgnum_1_c_system) || continue # Require that the crystal system remain the same
        sgnum_2_group = primitivize(spacegroup(sgnum_2))
        (filter(x->iszero(x.translation), sgnum_2_group) == sgnum_2_group) || continue # Only look at symmorphic space groups
        pg_2 = find_isomorphic_parent_pointgroup(sgnum_2_group)[1]
        (pg_1 == pg_2) || continue 
        push!(sgnum1_vec, sgnum_2)
    end
    !(isempty(sgnum1_vec)) || continue
    any(x -> x in filtered_sgnums_tr, sgnum1_vec) || continue
    push!(sgnum_pairs_primitive_tr, sgnum_1 => sgnum1_vec)
end

In [95]:
#filter(x -> !(x in keys(sgnum_pairs_primitive_tr)), keys(sgnum_pairs_primitive))

In [96]:
#filter(x -> !(x in keys(sgnum_pairs_primitive)), keys(sgnum_pairs_primitive_tr))

In [108]:
# If the original spacegroup is in the value of the (key, value) pair, we obviously just replace the value with a one element vector [key]
for (key, value) in sgnum_pairs_primitive
    if key in value
        sgnum_pairs_primitive[key] = [key]
    end
end

more_than_one_dict = filter(x -> length(x[2])>1, sgnum_pairs_primitive)

for (sgnum_1, sgnum_2v) in more_than_one_dict
    sg1_primitivized = primitivize(spacegroup(sgnum_1))
    sgnum_1_symmorphic_elements = filter(x->iszero(x.translation), sg1_primitivized)
    for sgnum_2 in sgnum_2v
        sg2_primitivized = primitivize(spacegroup(sgnum_2))
        (filter(x -> x in sg2_primitivized, sgnum_1_symmorphic_elements) == sgnum_1_symmorphic_elements) || continue
        sgnum_pairs_primitive[sgnum_1] = [sgnum_2]
    end
end

196
197
196
197
202
204


In [110]:
# If the original spacegroup is in the value of the (key, value) pair, we obviously just replace the value with a one element vector [key]
for (key, value) in sgnum_pairs_primitive_tr
    if key in value
        sgnum_pairs_primitive_tr[key] = [key]
    end
end
more_than_one_dict = filter(x -> length(x[2])>1, sgnum_pairs_primitive_tr)
for (sgnum_1, sgnum_2v) in more_than_one_dict
    sg1_primitivized = primitivize(spacegroup(sgnum_1))
    sgnum_1_symmorphic_elements = filter(x->iszero(x.translation), sg1_primitivized)
    for sgnum_2 in sgnum_2v
        sg2_primitivized = primitivize(spacegroup(sgnum_2))
        (filter(x -> x in sg2_primitivized, sgnum_1_symmorphic_elements) == sgnum_1_symmorphic_elements) || continue
        sgnum_pairs_primitive_tr[sgnum_1] = [sgnum_2]
    end
end

196
197
196
197
202
204


### Below, we find the maximum and minimum irrep dimensions for the spacegroup we reduce to after additiion of the defect. We do this both for spacegroups filtered in the presence of time reversal and those without time reversal

In [126]:
for (key, val) in  sgnum_pairs_primitive
    sgnum_1 = key
    sgnum_2 = val[1]
    irreps_at_gamma = lgirreps(sgnum_2)["Γ"]
    irreps_at_gamma_tr = realify(irreps_at_gamma)
    irrep_dims = [first(size(first(irrep_at_gamma.matrices))) for irrep_at_gamma in irreps_at_gamma]
    irrep_dims_tr = [first(size(first(irrep_at_gamma_tr.matrices))) for irrep_at_gamma_tr in irreps_at_gamma_tr]
    println("Min dimension (without tr): ", minimum(irrep_dims), "  ", "Max dimension (without tr): ", maximum(irrep_dims))
    println("Min dimension (with tr): ", minimum(irrep_dims_tr), "  ", "Max dimension (with tr): ", maximum(irrep_dims_tr), "\n")
end

Min dimension (without tr): 1  Max dimension (without tr): 2
Min dimension (with tr): 1  Max dimension (with tr): 2

Min dimension (without tr): 1  Max dimension (without tr): 3
Min dimension (with tr): 1  Max dimension (with tr): 3

Min dimension (without tr): 1  Max dimension (without tr): 3
Min dimension (with tr): 1  Max dimension (with tr): 3

Min dimension (without tr): 1  Max dimension (without tr): 3
Min dimension (with tr): 1  Max dimension (with tr): 3

Min dimension (without tr): 1  Max dimension (without tr): 2
Min dimension (with tr): 1  Max dimension (with tr): 2

Min dimension (without tr): 1  Max dimension (without tr): 2
Min dimension (with tr): 1  Max dimension (with tr): 2

Min dimension (without tr): 1  Max dimension (without tr): 2
Min dimension (with tr): 1  Max dimension (with tr): 2

Min dimension (without tr): 1  Max dimension (without tr): 2
Min dimension (with tr): 1  Max dimension (with tr): 2

Min dimension (without tr): 1  Max dimension (without tr): 3
Min

In [127]:
for (key, val) in  sgnum_pairs_primitive_tr
    sgnum_1 = key
    sgnum_2 = val[1]
    irreps_at_gamma = lgirreps(sgnum_2)["Γ"]
    irreps_at_gamma_tr = realify(irreps_at_gamma)
    irrep_dims = [first(size(first(irrep_at_gamma.matrices))) for irrep_at_gamma in irreps_at_gamma]
    irrep_dims_tr = [first(size(first(irrep_at_gamma_tr.matrices))) for irrep_at_gamma_tr in irreps_at_gamma_tr]
    println("Min dimension (without tr): ", minimum(irrep_dims), "  ", "Max dimension (without tr): ", maximum(irrep_dims))
    println("Min dimension (with tr): ", minimum(irrep_dims_tr), "  ", "Max dimension (with tr): ", maximum(irrep_dims_tr), "\n")
end

Min dimension (without tr): 1  Max dimension (without tr): 1
Min dimension (with tr): 1  Max dimension (with tr): 2

Min dimension (without tr): 1  Max dimension (without tr): 2
Min dimension (with tr): 1  Max dimension (with tr): 2

Min dimension (without tr): 1  Max dimension (without tr): 1
Min dimension (with tr): 1  Max dimension (with tr): 2

Min dimension (without tr): 1  Max dimension (without tr): 1
Min dimension (with tr): 1  Max dimension (with tr): 2

Min dimension (without tr): 1  Max dimension (without tr): 3
Min dimension (with tr): 1  Max dimension (with tr): 3

Min dimension (without tr): 1  Max dimension (without tr): 3
Min dimension (with tr): 1  Max dimension (with tr): 3

Min dimension (without tr): 1  Max dimension (without tr): 3
Min dimension (with tr): 1  Max dimension (with tr): 3

Min dimension (without tr): 1  Max dimension (without tr): 2
Min dimension (with tr): 1  Max dimension (with tr): 2

Min dimension (without tr): 1  Max dimension (without tr): 2
Min